In [ ]:
import pandas as pd
import os
import glob, os
from tqdm.notebook import tqdm
import json
import plotly.express as px
import geopandas as gpd
import numpy as np

%matplotlib inline
pd.set_option('display.max_columns', None)

# Datos electorales

In [ ]:
data_path = "../data/raw/resultados_electorales/congreso_por_municipio/"
df = None
for file in tqdm(glob.glob(data_path + "*.xlsx")):
    df_single = pd.read_excel(file, header=1, skiprows=4)
    year = os.path.splitext(file)[0].split("/")[-1][3:7]
    month = os.path.splitext(file)[0].split("/")[-1][7:9]
    df_single["Año"] = int(year)
    df_single["Mes"] = int(month)
    df_single["añomes"] = year+month
    print("yearmonth: ", year+month)
    if df is None:
        df = df_single
    else:
        df = pd.concat([df, df_single], ignore_index=True, sort=False)
        
    # QUITAR
    break

In [ ]:
df.columns

In [ ]:
# limpiar columnas index y rellenar NA
df = df[df.columns.drop(list(df.filter(regex='Unnamed')))]
df = df.fillna(0)
# eliminar columnas con menso de x votos
#df_sum = pd.DataFrame(df.sum(axis = 0, skipna = True, numeric_only=True))
#df_sum[df_sum[0] == 0].index

In [ ]:
# crear codigo ine
df["CODIGOINE"] = df["Código de Provincia"].astype(int).astype(str) + df["Código de Municipio"].astype(int).astype(str).str.zfill(3)

In [ ]:
df.tail(5)

In [ ]:
df.shape

In [ ]:
df.dtypes

# Mapa España por munipios

In [ ]:
#map_path = "../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN.shp"
#geodf = gpd.read_file(map_path)
#geodf_reduced = geodf[["CODIGOINE","geometry"]]
#geojson_save_path = "../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN.geojson"
#guardar
#geodf_reduced.to_file(geojson_path, driver = "GeoJSON")

In [ ]:
geojson_path = "../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN_reduced.geojson"

In [ ]:
#cargar
with open(geojson_path) as geofile:
    geojson = json.load(geofile)

# Gráfico

In [ ]:
df_filtered = df[df["añomes"]=="199603"][["Número de mesas", "CODIGOINE"]][:10]

In [ ]:
df_filtered.shape

In [ ]:
fig = px.choropleth_mapbox(df_filtered, geojson=geojson, locations='CODIGOINE', featureidkey="properties.CODIGOINE", 
                           color='Número de mesas', mapbox_style="carto-positron", 
                           center={"lat": 40.463667, "lon": -3.74922}, zoom=9)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

In [ ]:
df_filtered.head()

In [ ]:
df_p.head()

In [ ]:
geojson["features"][0]

In [ ]:
geojson_p["features"][0]

In [ ]:
df_p = px.data.election()
geojson_p = px.data.election_geojson()

fig = px.choropleth_mapbox(df_p, geojson=geojson_p, color="winner",
                           locations="district", featureidkey="properties.district",
                           center={"lat": 45.5517, "lon": -73.7073},
                           mapbox_style="carto-positron", zoom=9)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()